# 1 - Raw Data Preparation - All Families

This notebook implements Steps **01–04** of the experimental pipeline:

- **01 — Import raw data**
- **02 — Merge / harmonize data**
- **03 — Build the canonical pair-level dataset**
- **04 — Audit raw and processed datasets**

It uses one common procedure for the four Yamanishi ligand families:

- Enzyme
- GPCR
- Ion Channel
- Nuclear Receptor

The raw filenames are generated from a **family prefix dictionary** and a **suffix dictionary**.  
The notebook does **not** scan the full `data/raw` folder.

Legacy matrix naming from the original notebooks is preserved:

- `Y` = adjacency / interaction matrix
- `St` = compound similarity
- `Sd` = protein similarity

No cross-validation, matrix factorization, augmentation, or DNN training is performed here.

In [1]:
from pathlib import Path
import json
import hashlib

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
AGG_DIR = PROJECT_ROOT / "data" / "agg"
SPLIT_DIR = PROJECT_ROOT / "data" / "split"

AGG_DIR.mkdir(parents=True, exist_ok=True)

print("Project root :", PROJECT_ROOT)
print("Raw folder   :", RAW_DIR)
print("Output folder:", AGG_DIR)
print("Split folder :", SPLIT_DIR, "(not used in Notebook 1)")

assert RAW_DIR.exists(), f"Raw folder not found: {RAW_DIR}"

Project root : c:\Users\riskf\OneDrive\A-DTI2026
Raw folder   : c:\Users\riskf\OneDrive\A-DTI2026\data\raw
Output folder: c:\Users\riskf\OneDrive\A-DTI2026\data\agg
Split folder : c:\Users\riskf\OneDrive\A-DTI2026\data\split (not used in Notebook 1)


## 1. File dictionaries

The filenames follow the same construction used in the original family notebooks.

For example, Enzyme becomes:

```text
e + _admat_dgc.txt  -> e_admat_dgc.txt
e + _simmat_dc.txt  -> e_simmat_dc.txt
e + _simmat_dg.txt  -> e_simmat_dg.txt
```

In [2]:
FAMILIES = {
    "enzyme": {
        "display": "Enzyme",
        "prefix": "e",
        "pair_prefix": "ENZ",
    },
    "gpcr": {
        "display": "GPCR",
        "prefix": "gpcr",
        "pair_prefix": "GPCR",
    },
    "ion_channel": {
        "display": "Ion Channel",
        "prefix": "ic",
        "pair_prefix": "IC",
    },
    "nuclear_receptor": {
        "display": "Nuclear Receptor",
        "prefix": "nr",
        "pair_prefix": "NR",
    },
}

SUFFIXES = {
    "Y":  "_admat_dgc.txt",
    "St": "_simmat_dc.txt",   # compound similarity
    "Sd": "_simmat_dg.txt",   # protein similarity
}

FILES = {
    family: {
        role: RAW_DIR / f"{cfg['prefix']}{suffix}"
        for role, suffix in SUFFIXES.items()
    }
    for family, cfg in FAMILIES.items()
}

for family, cfg in FAMILIES.items():
    print(f"\n{cfg['display']}")
    for role in ["Y", "St", "Sd"]:
        print(f"  {role}: {FILES[family][role].name}")


Enzyme
  Y: e_admat_dgc.txt
  St: e_simmat_dc.txt
  Sd: e_simmat_dg.txt

GPCR
  Y: gpcr_admat_dgc.txt
  St: gpcr_simmat_dc.txt
  Sd: gpcr_simmat_dg.txt

Ion Channel
  Y: ic_admat_dgc.txt
  St: ic_simmat_dc.txt
  Sd: ic_simmat_dg.txt

Nuclear Receptor
  Y: nr_admat_dgc.txt
  St: nr_simmat_dc.txt
  Sd: nr_simmat_dg.txt


## 2. Confirm the 12 required raw files

This checks only the expected filenames generated from the dictionaries above.  
It does not inspect unrelated files or folders under `data/raw`.

In [3]:
missing = []

for family, cfg in FAMILIES.items():
    for role in ["Y", "St", "Sd"]:
        path = FILES[family][role]
        if not path.exists():
            missing.append((cfg["display"], role, path.name))

if missing:
    print("Missing raw files:")
    for family, role, filename in missing:
        print(f"  {family:18s} {role:2s} -> {filename}")
    raise FileNotFoundError(
        "One or more expected Yamanishi files are missing from data/raw."
    )

print("PASS - all 12 expected raw files were found.")

PASS - all 12 expected raw files were found.


# Step 01 — Import raw data

The three matrices for each family are loaded directly from the expected files.

In [4]:
def load_matrix(path):
    df = pd.read_csv(path, sep="\t", index_col=0)

    # Remove only completely empty rows or columns.
    df = df.dropna(axis=0, how="all").dropna(axis=1, how="all")

    # Matrix contents must be numeric.
    df = df.apply(pd.to_numeric, errors="raise")

    # Keep identifiers as strings.
    df.index = df.index.astype(str)
    df.columns = df.columns.astype(str)

    return df


DATA = {}

for family, cfg in FAMILIES.items():
    DATA[family] = {
        "Y": load_matrix(FILES[family]["Y"]),
        "St": load_matrix(FILES[family]["St"]),
        "Sd": load_matrix(FILES[family]["Sd"]),
    }

    Y = DATA[family]["Y"]
    St = DATA[family]["St"]
    Sd = DATA[family]["Sd"]

    print(
        f"{cfg['display']:<18} "
        f"Y={Y.shape} | "
        f"St compound={St.shape} | "
        f"Sd protein={Sd.shape}"
    )

Enzyme             Y=(664, 445) | St compound=(445, 445) | Sd protein=(664, 664)
GPCR               Y=(95, 223) | St compound=(223, 223) | Sd protein=(95, 95)
Ion Channel        Y=(204, 210) | St compound=(210, 210) | Sd protein=(204, 204)
Nuclear Receptor   Y=(26, 54) | St compound=(54, 54) | Sd protein=(26, 26)


# Step 02 — Harmonize and validate matrices


where rows of `Y` are compounds/drugs and columns are proteins/targets.

The similarity matrices are reordered to match the row and column identifiers of `Y`.
No matrix values are transformed.

In [6]:
for family, cfg in FAMILIES.items():

    Y = DATA[family]["Y"]
    St = DATA[family]["St"]   # compound similarity
    Sd = DATA[family]["Sd"]   # protein similarity

    n_proteins, n_compounds = Y.shape

    # Compound similarity corresponds to the COLUMNS of Y
    assert St.shape == (n_compounds, n_compounds), (
        f"{cfg['display']}: St shape {St.shape} does not match "
        f"{n_compounds} compound columns in Y."
    )

    # Protein similarity corresponds to the ROWS of Y
    assert Sd.shape == (n_proteins, n_proteins), (
        f"{cfg['display']}: Sd shape {Sd.shape} does not match "
        f"{n_proteins} protein rows in Y."
    )

    # Y must contain only binary interaction labels
    y_values = set(np.unique(Y.to_numpy()))

    assert y_values.issubset({0, 1}), (
        f"{cfg['display']}: Y contains values outside {{0,1}}: "
        f"{sorted(y_values)}"
    )

    # Y rows = proteins
    assert Y.index.is_unique, (
        f"{cfg['display']}: duplicate protein IDs in Y."
    )

    # Y columns = compounds
    assert Y.columns.is_unique, (
        f"{cfg['display']}: duplicate compound IDs in Y."
    )

    # Check that the entities in Y exist in the similarity matrices
    missing_proteins = set(Y.index) - set(Sd.index)
    missing_compounds = set(Y.columns) - set(St.index)

    assert not missing_proteins, (
        f"{cfg['display']}: proteins in Y missing from Sd: "
        f"{sorted(missing_proteins)[:10]}"
    )

    assert not missing_compounds, (
        f"{cfg['display']}: compounds in Y missing from St: "
        f"{sorted(missing_compounds)[:10]}"
    )

    # Reorder similarity matrices to exactly match Y
    Sd = Sd.loc[Y.index, Y.index]
    St = St.loc[Y.columns, Y.columns]

    DATA[family]["Sd"] = Sd
    DATA[family]["St"] = St

    print(
        f"PASS - {cfg['display']} | "
        f"Y={Y.shape}, "
        f"proteins={n_proteins}, "
        f"compounds={n_compounds}"
    )

PASS - Enzyme | Y=(664, 445), proteins=664, compounds=445
PASS - GPCR | Y=(95, 223), proteins=95, compounds=223
PASS - Ion Channel | Y=(204, 210), proteins=204, compounds=210
PASS - Nuclear Receptor | Y=(26, 54), proteins=26, compounds=54


# Step 03 — Build canonical pair-level datasets

Each cell of `Y` becomes one observation with a permanent `pair_id`.



In [7]:
PAIR_TABLES = {}

for family, cfg in FAMILIES.items():

    Y = DATA[family]["Y"]

    records = []
    pair_number = 0

    # Y rows = proteins
    for protein_index, protein_id in enumerate(Y.index):

        # Y columns = compounds
        for compound_index, compound_id in enumerate(Y.columns):

            pair_number += 1

            records.append({
                "pair_id": f"{cfg['pair_prefix']}_{pair_number:08d}",
                "family": family,
                "compound_id": compound_id,
                "protein_id": protein_id,
                "compound_index": compound_index,
                "protein_index": protein_index,
                "y_row": protein_index,
                "y_col": compound_index,
                "y": int(Y.iloc[protein_index, compound_index]),
            })

    pairs = pd.DataFrame(records)

    assert len(pairs) == Y.size
    assert pairs["pair_id"].is_unique
    assert not pairs[
        ["compound_id", "protein_id"]
    ].duplicated().any()

    PAIR_TABLES[family] = pairs

    print(
        f"{cfg['display']:<18} "
        f"pairs={len(pairs):,} | "
        f"positive={(pairs['y'] == 1).sum():,} | "
        f"non-interaction={(pairs['y'] == 0).sum():,}"
    )

Enzyme             pairs=295,480 | positive=2,926 | non-interaction=292,554
GPCR               pairs=21,185 | positive=635 | non-interaction=20,550
Ion Channel        pairs=42,840 | positive=1,476 | non-interaction=41,364
Nuclear Receptor   pairs=1,404 | positive=90 | non-interaction=1,314


# Step 04 — Audit the reconstructed datasets

In [8]:
audit_rows = []

for family, cfg in FAMILIES.items():
    Y = DATA[family]["Y"]
    St = DATA[family]["St"]
    Sd = DATA[family]["Sd"]
    pairs = PAIR_TABLES[family]

    n_pairs = len(pairs)
    n_positive = int((pairs["y"] == 1).sum())
    n_non_interaction = int((pairs["y"] == 0).sum())

    audit_rows.append({
        "family": cfg["display"],
        "n_compounds": Y.shape[0],
        "n_proteins": Y.shape[1],
        "n_pairs": n_pairs,
        "n_positive": n_positive,
        "n_non_interaction": n_non_interaction,
        "positive_rate": n_positive / n_pairs,
        "non_interaction_rate": n_non_interaction / n_pairs,
        "Y_shape": str(Y.shape),
        "St_compound_shape": str(St.shape),
        "Sd_protein_shape": str(Sd.shape),
        "duplicate_pairs": int(
            pairs[["compound_id", "protein_id"]].duplicated().sum()
        ),
        "missing_values": int(pairs.isna().sum().sum()),
    })

AUDIT = pd.DataFrame(audit_rows)
display(AUDIT)

,family,n_compounds,n_proteins,n_pairs,n_positive,n_non_interaction,positive_rate,non_interaction_rate,Y_shape,St_compound_shape,Sd_protein_shape,duplicate_pairs,missing_values
0,Enzyme,664,445,295480,2926,292554,0.009903,0.990097,"(664, 445)","(445, 445)","(664, 664)",0,0
1,GPCR,95,223,21185,635,20550,0.029974,0.970026,"(95, 223)","(223, 223)","(95, 95)",0,0
2,Ion Channel,204,210,42840,1476,41364,0.034454,0.965546,"(204, 210)","(210, 210)","(204, 204)",0,0
3,Nuclear Receptor,26,54,1404,90,1314,0.064103,0.935897,"(26, 54)","(54, 54)","(26, 26)",0,0


In [9]:
# Save audit statistics
STATS_DIR = PROJECT_ROOT / "Stats"
STATS_DIR.mkdir(parents=True, exist_ok=True)

audit_file = STATS_DIR / "dataset_audit.xlsx"

AUDIT.to_excel(
    audit_file,
    index=False,
    sheet_name="Dataset Audit"
)

print(f"Audit saved to: {audit_file}")

Audit saved to: c:\Users\riskf\OneDrive\A-DTI2026\Stats\dataset_audit.xlsx


## Save standardized outputs

For each family, Notebook 1 writes:

```text
data/agg/<family>/
├── Y.csv
├── St_compound.csv
├── Sd_protein.csv
├── pairs.csv
└── source_manifest.json
```

It also writes:

```text
data/agg/dataset_audit.csv
```

In [11]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


for family, cfg in FAMILIES.items():
    out_dir = AGG_DIR / family
    out_dir.mkdir(parents=True, exist_ok=True)

    DATA[family]["Y"].to_csv(out_dir / "Y.csv")
    DATA[family]["St"].to_csv(out_dir / "St_compound.csv")
    DATA[family]["Sd"].to_csv(out_dir / "Sd_protein.csv")
    PAIR_TABLES[family].to_csv(out_dir / "pairs.csv", index=False)

    manifest = {
        "family": family,
        "display_name": cfg["display"],
        "legacy_notation": {
            "Y": "drug-target interaction matrix",
            "St": "compound similarity",
            "Sd": "protein similarity",
        },
        "source_files": {
            role: {
                "filename": FILES[family][role].name,
                "sha256": sha256_file(FILES[family][role]),
            }
            for role in ["Y", "St", "Sd"]
        },
        "shapes": {
            "Y": list(DATA[family]["Y"].shape),
            "St_compound": list(DATA[family]["St"].shape),
            "Sd_protein": list(DATA[family]["Sd"].shape),
            "pairs": list(PAIR_TABLES[family].shape),
        },
    }

    with open(out_dir / "source_manifest.json", "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)

AUDIT.to_csv(STATS_DIR / "dataset_audit.csv", index=False)

print("Saved outputs to:", AGG_DIR)

Saved outputs to: c:\Users\riskf\OneDrive\A-DTI2026\data\agg


## Final integrity check

Notebook 1 is complete only if all four families pass.

In [13]:
for family, cfg in FAMILIES.items():
    out_dir = AGG_DIR / family

    required = [
        "Y.csv",
        "St_compound.csv",
        "Sd_protein.csv",
        "pairs.csv",
        "source_manifest.json",
    ]

    for filename in required:
        assert (out_dir / filename).exists(), (
            f"{cfg['display']}: missing output {filename}"
        )

    pairs = pd.read_csv(out_dir / "pairs.csv")

    assert pairs["pair_id"].is_unique
    assert set(pairs["y"].unique()).issubset({0, 1})

    print(f"PASS - {cfg['display']}")

assert (STATS_DIR / "dataset_audit.csv").exists()

print("\nNotebook 1 completed successfully.")
print("Next: Notebook 2 - fixed stratified 5-fold outer splits.")

PASS - Enzyme
PASS - GPCR
PASS - Ion Channel
PASS - Nuclear Receptor

Notebook 1 completed successfully.
Next: Notebook 2 - fixed stratified 5-fold outer splits.
